# Day 10 · Exercise 5: Template-Driven LLM Call

**What you'll build:** `call_with_template(template_str: str, variables: dict, system_prompt: str, model: str) -> str` — a function that renders a `string.Template` with a variable dict to produce a filled prompt string, then passes that string as the user message in an `ollama.chat()` call and returns the model's reply.

**Why it matters:** Keeping prompt construction (render) separate from model invocation (call) is the core discipline of maintainable AI engineering — it lets you unit-test your prompts without touching the model and swap templates or models independently.

## Your Implementation

In [ ]:
import ollama
from string import Template


def call_with_template(
    template_str: str,
    variables: dict,
    system_prompt: str,
    model: str,
) -> str:
    """Render a template string with variables, then call an Ollama model.

    This implements the render-then-call pattern in two explicit steps:
    1. Render: substitute `variables` into `template_str` to produce a
       filled prompt string (using string.Template).
    2. Call: pass the filled string as the user message in ollama.chat(),
       optionally preceded by a system message.

    Args:
        template_str:  A string.Template-style template, e.g.
                       "Summarize in $max_sentences sentences.\n\n$text"
        variables:     A dict mapping placeholder names to their values, e.g.
                       {"text": "...", "max_sentences": "2"}
        system_prompt: Content for the system message prepended before the
                       user turn. Pass an empty string to omit it.
        model:         The Ollama model name to use, e.g. "llama3.2".

    Returns:
        The model's reply as a plain string.

    Example:
        reply = call_with_template(
            "Translate to $language: $text",
            {"language": "French", "text": "Hello, world."},
            system_prompt="You are a concise translator.",
            model="llama3.2",
        )
        # reply → "Bonjour, le monde."
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 4

    # Check 1: function is defined and callable
    try:
        assert callable(call_with_template), 'call_with_template is not callable'
        print(f'{_PASS} Check 1/{total}: call_with_template is defined and callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: render step — call_with_template performs substitution correctly.
    # We verify this by intercepting ollama.chat and inspecting the messages list
    # it receives, so the check tests the student's render logic without requiring
    # Ollama to be running.
    try:
        import unittest.mock as mock
        import ollama as _ollama

        captured_messages = []

        def _fake_chat(model, messages):
            captured_messages.extend(messages)
            return {"message": {"content": "stub reply"}}

        with mock.patch.object(_ollama, 'chat', side_effect=_fake_chat):
            call_with_template(
                'Hello, $name! You have $count new messages.',
                {'name': 'Alice', 'count': '3'},
                system_prompt='',
                model='llama3.2',
            )

        user_msgs = [m for m in captured_messages if m.get('role') == 'user']
        assert user_msgs, 'no user message was passed to ollama.chat'
        content = user_msgs[0]['content']
        assert 'Alice' in content, f'"Alice" not found in rendered user message: {content!r}'
        assert '3' in content, f'"3" not found in rendered user message: {content!r}'
        assert '$name' not in content, f'$name placeholder was not substituted: {content!r}'
        assert '$count' not in content, f'$count placeholder was not substituted: {content!r}'
        print(f'{_PASS} Check 2/{total}: render step substitutes variables correctly into the user message')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: full round-trip — call_with_template returns a non-empty string
    # Requires Ollama to be running with a model available.
    try:
        reply = call_with_template(
            'In one sentence, what is $topic?',
            {'topic': 'the Python programming language'},
            system_prompt='You are a concise assistant. Reply in exactly one sentence.',
            model='llama3.2',
        )
        assert isinstance(reply, str), f'expected str, got {type(reply).__name__}'
        assert len(reply.strip()) > 0, 'reply is empty'
        print(f'{_PASS} Check 3/{total}: call_with_template returned a non-empty string')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: missing placeholder raises KeyError (substitution error), not a silent failure
    try:
        raised = False
        try:
            call_with_template(
                'Translate $text to $language.',
                {'text': 'Hello'},   # 'language' is missing — must raise
                system_prompt='',
                model='llama3.2',
            )
        except (KeyError, ValueError):
            raised = True
        assert raised, 'expected KeyError or ValueError for missing placeholder, got nothing'
        print(f'{_PASS} Check 4/{total}: missing placeholder raises an error (not silent)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Add a `logger.debug()` call between the render step and the `ollama.chat()` call that logs the full rendered prompt string. Then re-run a call and inspect the output — this is what your logs would contain in production, which is the first thing you reach for when a model gives an unexpected answer.

This foreshadows Day 11's structured logging patterns, where every LLM call emits a JSON log line containing the model, the prompt length, and the latency.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama
from string import Template


def call_with_template(
    template_str: str,
    variables: dict,
    system_prompt: str,
    model: str,
) -> str:
    # ── Step 1: render — template + variables → filled string ──
    tmpl = Template(template_str)
    user_message = tmpl.substitute(**variables)  # raises KeyError if a placeholder is missing

    # ── Step 2: build messages list ────────────────────────────
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    # ── Step 3: call — filled string → model response ──────────
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]
```

**Why this works:** `Template(template_str).substitute(**variables)` is the entire render step — it produces a plain Python string that contains exactly what the model will read, with no model involved. That string is the seam between prompt construction and model invocation: you could log it, assert on it in a test, or hand it to a different model entirely. Calling `substitute` (not `safe_substitute`) is intentional — it raises `KeyError` immediately when a placeholder is missing, giving you a fast, clear error instead of silently sending a broken prompt.
</details>